# Granite Peak CRE: 01 Generate Synthetic Data

Creates the entire synthetic dataset for the Granite Peak CRE build:

1. **ERP tables** shaped like Yardi Voyager: `property`, `unit`, `tenant`, `commlease`, `lease_charges`, `gl_actuals`, plus `fund`
2. **CDC landing files** in Qlik Replicate style (`header__operation`, `header__timestamp`) with 24+ months of history
3. **Deal pipeline** (`deal`, `deal_comps`) including deal GP-ACQ-014
4. **Lease PDFs** for the risk-weighted subset (~60-100) with ground-truth JSON
5. **Deal documents** (~10-15 offering memoranda and rent rolls)

All data is synthetic. A fixed random seed ensures reproducibility.

In [0]:
dbutils.widgets.text("catalog", "smmorgan_catalog", "Catalog")
dbutils.widgets.text("schema", "granite_commercial_real_estate", "Schema")

catalog = dbutils.widgets.get("catalog").strip()
schema  = dbutils.widgets.get("schema").strip()
assert catalog and schema, "Set the 'catalog' and 'schema' widgets."

VOL_LANDING   = f"/Volumes/{catalog}/{schema}/landing"
VOL_ARTIFACTS = f"/Volumes/{catalog}/{schema}/artifacts"
print(f"Landing:   {VOL_LANDING}")
print(f"Artifacts: {VOL_ARTIFACTS}")

In [0]:
%pip install fpdf2 -q
dbutils.library.restartPython()

In [0]:
import json, os, random, math, datetime, hashlib, uuid, itertools
from collections import defaultdict
from datetime import date, timedelta
from fpdf import FPDF
import pandas as pd

# Re-read widgets after restartPython
catalog = dbutils.widgets.get("catalog").strip()
schema  = dbutils.widgets.get("schema").strip()
VOL_LANDING   = f"/Volumes/{catalog}/{schema}/landing"
VOL_ARTIFACTS = f"/Volumes/{catalog}/{schema}/artifacts"

SEED = 42
random.seed(SEED)
AS_OF = date(2026, 10, 1)  # reference date for expiry calculations

# ~8 Western / Mountain-West MSAs with center coords and radius for scatter
MSAS = {
    "Salt Lake City": {"state": "UT", "lat": 40.7608, "lon": -111.8910, "r": 0.12},
    "Denver":         {"state": "CO", "lat": 39.7392, "lon": -104.9903, "r": 0.15},
    "Phoenix":        {"state": "AZ", "lat": 33.4484, "lon": -112.0740, "r": 0.18},
    "Boise":          {"state": "ID", "lat": 43.6150, "lon": -116.2023, "r": 0.08},
    "Las Vegas":      {"state": "NV", "lat": 36.1699, "lon": -115.1398, "r": 0.12},
    "Portland":       {"state": "OR", "lat": 45.5152, "lon": -122.6784, "r": 0.12},
    "Seattle":        {"state": "WA", "lat": 47.6062, "lon": -122.3321, "r": 0.12},
    "Albuquerque":    {"state": "NM", "lat": 35.0844, "lon": -106.6504, "r": 0.10},
}

PROPERTY_TYPES = ["Office", "Retail", "Industrial"]
FUNDS = ["GP Fund I", "GP Fund II", "GP Fund III", "GP Opportunity IV"]

# Commercial tenant name components
TENANT_PREFIXES = ["Apex", "Summit", "Pinnacle", "Redstone", "Basin", "Crest", "Valley", "Mesa",
                   "Canyon", "Ridge", "Horizon", "Frontier", "Cascade", "Sierra", "Aspen",
                   "Copper", "Silver", "Iron", "Granite", "Eagle", "Falcon", "Arrow",
                   "Peak", "Blue", "Cedar", "Pine", "Stone", "Brook", "Lake", "West"]
TENANT_SUFFIXES_OFFICE = ["Analytics", "Consulting", "Financial", "Legal Group", "Technologies",
                          "Advisors", "Capital", "Partners", "Solutions", "Management"]
TENANT_SUFFIXES_RETAIL = ["Coffee Co", "Fitness", "Pharmacy", "Market", "Outfitters",
                          "Brewery", "Kitchen", "Salon", "Optics", "Pet Supply"]
TENANT_SUFFIXES_INDUSTRIAL = ["Logistics", "Distribution", "Manufacturing", "Freight",
                              "Warehouse Co", "Supply Chain", "Materials", "Fabrication"]

EXTERIOR_STREETS = ["Main St", "Center St", "Broadway", "Market St", "Park Ave",
                    "Industrial Blvd", "Commerce Dr", "Tech Way", "Gateway Rd",
                    "Enterprise Dr", "Lakefront Dr", "Sierra Ave", "Canyon Blvd"]

print(f"Seed: {SEED}  |  As-of date: {AS_OF}  |  MSAs: {len(MSAS)}  |  Funds: {len(FUNDS)}")

In [0]:
def _scatter(center_lat, center_lon, r):
    """Random point within r degrees of center."""
    angle = random.uniform(0, 2 * math.pi)
    dist  = random.uniform(0, r)
    return round(center_lat + dist * math.sin(angle), 6), round(center_lon + dist * math.cos(angle), 6)

def _street_addr():
    return f"{random.randint(100, 9999)} {random.choice(EXTERIOR_STREETS)}"

# --- Funds ---
funds = [{"fund_id": f"GP-FND-{i+1:03d}", "fund_name": f} for i, f in enumerate(FUNDS)]

# --- Properties (target ~120, split roughly evenly across MSAs and types) ---
properties = []
prop_id_seq = 0
for msa_name, msa in MSAS.items():
    n_props = random.randint(13, 17)  # ~15 per MSA -> ~120 total
    for _ in range(n_props):
        prop_id_seq += 1
        ptype = random.choice(PROPERTY_TYPES)
        lat, lon = _scatter(msa["lat"], msa["lon"], msa["r"])
        sqft_ranges = {"Office": (20_000, 250_000), "Retail": (8_000, 120_000), "Industrial": (40_000, 500_000)}
        lo, hi = sqft_ranges[ptype]
        sqft = random.randint(lo // 1000, hi // 1000) * 1000
        properties.append({
            "property_id": f"GP-PROP-{prop_id_seq:04d}",
            "property_name": f"Granite {msa_name} {ptype} {prop_id_seq}",
            "property_type": ptype,
            "address": _street_addr(),
            "city": msa_name,
            "state": msa["state"],
            "zip_code": f"{random.randint(80000, 99999)}",
            "lat": lat,
            "lon": lon,
            "year_built": random.randint(1970, 2022),
            "total_sqft": sqft,
            "floors": random.randint(1, 3) if ptype == "Industrial" else random.randint(1, 12),
            "fund_id": random.choice(funds)["fund_id"],
            "msa": msa_name,
        })

# --- Units ---
units = []
unit_seq = 0
for prop in properties:
    n_units = random.randint(2, 8) if prop["property_type"] != "Industrial" else random.randint(1, 4)
    unit_sqft = prop["total_sqft"] // n_units
    for u in range(n_units):
        unit_seq += 1
        units.append({
            "unit_id": f"GP-UNIT-{unit_seq:05d}",
            "property_id": prop["property_id"],
            "unit_number": f"{random.randint(1, prop['floors'])}{chr(65 + u)}",
            "floor": random.randint(1, prop["floors"]),
            "sqft": unit_sqft + random.randint(-500, 500),
            "status": random.choices(["Occupied", "Vacant"], weights=[85, 15])[0],
        })

print(f"Properties: {len(properties)}  |  Units: {len(units)}  |  Funds: {len(funds)}")
print(f"By type: { {t: sum(1 for p in properties if p['property_type']==t) for t in PROPERTY_TYPES} }")
print(f"By MSA:  { {m: sum(1 for p in properties if p['msa']==m) for m in MSAS} }")

In [0]:
# --- Tenants ---
tenants = []
tenant_seq = 0
occupied_units = [u for u in units if u["status"] == "Occupied"]

def _tenant_name(ptype):
    prefix = random.choice(TENANT_PREFIXES)
    suffix_map = {"Office": TENANT_SUFFIXES_OFFICE, "Retail": TENANT_SUFFIXES_RETAIL,
                  "Industrial": TENANT_SUFFIXES_INDUSTRIAL}
    return f"{prefix} {random.choice(suffix_map[ptype])}"

for u in occupied_units:
    tenant_seq += 1
    prop = next(p for p in properties if p["property_id"] == u["property_id"])
    tenants.append({
        "tenant_id": f"GP-TNT-{tenant_seq:05d}",
        "tenant_name": _tenant_name(prop["property_type"]),
        "industry": random.choice(["Technology", "Finance", "Healthcare", "Retail", "Logistics",
                                   "Legal", "Energy", "Food & Beverage", "Manufacturing"]),
        "credit_rating": random.choices(["AAA", "AA", "A", "BBB", "BB", "B"],
                                        weights=[5, 10, 25, 30, 20, 10])[0],
        "contact_name": f"Contact {tenant_seq}",
        "contact_email": f"contact{tenant_seq}@example.com",
        "contact_phone": f"555-{random.randint(1000,9999)}",
    })

# --- Commercial Leases ---
leases = []
lease_seq = 0
for i, u in enumerate(occupied_units):
    lease_seq += 1
    prop = next(p for p in properties if p["property_id"] == u["property_id"])
    tenant = tenants[i]
    # Lease start: 1-8 years ago; term: 3-15 years
    years_ago = random.randint(1, 8)
    term_years = random.choice([3, 5, 7, 10, 15])
    start = AS_OF - timedelta(days=years_ago * 365 + random.randint(0, 180))
    end   = start + timedelta(days=term_years * 365)
    psf   = {"Office": (18, 55), "Retail": (22, 65), "Industrial": (6, 18)}
    lo, hi = psf[prop["property_type"]]
    annual_psf = round(random.uniform(lo, hi), 2)
    abr = round(annual_psf * u["sqft"], 2)
    monthly = round(abr / 12, 2)

    esc_type = random.choice(["Fixed", "CPI", "Stepped"])
    renewal_ct = random.choice([0, 0, 1, 1, 2, 3]) if random.random() < 0.6 else 0
    has_co_ten = random.random() < 0.25 if prop["property_type"] == "Retail" else random.random() < 0.05
    has_term   = random.random() < 0.20
    has_excl   = random.random() < 0.15 if prop["property_type"] == "Retail" else False
    term_dt    = (start + timedelta(days=random.randint(365, term_years*365-180))).isoformat() if has_term else None

    leases.append({
        "lease_id": f"GP-LSE-{lease_seq:05d}",
        "property_id": prop["property_id"],
        "unit_id": u["unit_id"],
        "tenant_id": tenant["tenant_id"],
        "lease_start": start.isoformat(),
        "lease_end": end.isoformat(),
        "monthly_rent": monthly,
        "annual_base_rent": abr,
        "escalation_type": esc_type,
        "escalation_rate": round(random.uniform(0.02, 0.04), 4) if esc_type != "CPI" else None,
        "renewal_option_count": renewal_ct,
        "has_co_tenancy": has_co_ten,
        "co_tenancy_terms": f"Anchor tenant must maintain {random.randint(60,80)}% occupancy" if has_co_ten else None,
        "has_termination_right": has_term,
        "termination_right_date": term_dt,
        "has_exclusive_use": has_excl,
        "exclusive_use_terms": f"Exclusive {random.choice(['restaurant','gym','pharmacy','coffee'])} within property" if has_excl else None,
        "security_deposit": round(monthly * random.choice([1, 2, 3]), 2),
    })

print(f"Tenants: {len(tenants)}  |  Leases: {len(leases)}")
expiring_36mo = [l for l in leases if date.fromisoformat(l['lease_end']) <= AS_OF + timedelta(days=36*30)]
print(f"Leases expiring within 36 months: {len(expiring_36mo)}")
top25 = sorted(leases, key=lambda l: l['annual_base_rent'], reverse=True)[:25]
print(f"Top 25 by ABR, smallest ABR: ${top25[-1]['annual_base_rent']:,.0f}")

In [0]:
# --- lease_charges: one row per lease per month for past 24 months ---
lease_charges = []
charge_seq = 0
months_back = 24
for lease in leases:
    l_start = date.fromisoformat(lease["lease_start"])
    l_end   = date.fromisoformat(lease["lease_end"])
    for m in range(months_back):
        period = AS_OF - timedelta(days=30 * (months_back - m))
        if period < l_start or period > l_end:
            continue
        charge_seq += 1
        lease_charges.append({
            "charge_id": f"GP-CHG-{charge_seq:07d}",
            "lease_id": lease["lease_id"],
            "charge_period": period.isoformat()[:7],  # YYYY-MM
            "charge_type": "Base Rent",
            "amount": lease["monthly_rent"],
            "paid": random.choices([True, False], weights=[95, 5])[0],
        })

# --- gl_actuals: monthly property-level NOI entries ---
gl_actuals = []
gl_seq = 0
for prop in properties:
    prop_leases = [l for l in leases if l["property_id"] == prop["property_id"]]
    monthly_rev = sum(l["monthly_rent"] for l in prop_leases)
    for m in range(months_back):
        period = AS_OF - timedelta(days=30 * (months_back - m))
        gl_seq += 1
        rev = round(monthly_rev * random.uniform(0.92, 1.05), 2)
        opex = round(rev * random.uniform(0.30, 0.55), 2)
        gl_actuals.append({
            "gl_id": f"GP-GL-{gl_seq:07d}",
            "property_id": prop["property_id"],
            "period": period.isoformat()[:7],
            "revenue": rev,
            "operating_expense": opex,
            "noi": round(rev - opex, 2),
        })

print(f"Lease charges: {len(lease_charges):,}  |  GL actuals: {len(gl_actuals):,}")

In [0]:
def write_cdc_file(records, table_name, operation, ts, out_dir):
    """Write a single CDC JSON-lines file in Qlik Replicate style."""
    fname = f"{table_name}_{ts.strftime('%Y%m%d_%H%M%S')}_{operation}.jsonl"
    path = os.path.join(out_dir, fname)
    with open(path, "w") as f:
        for rec in records:
            row = {"header__operation": operation, "header__timestamp": ts.isoformat(), **rec}
            f.write(json.dumps(row, default=str) + "\n")
    return len(records), fname

cdc_dir = f"{VOL_LANDING}/erp_cdc"
os.makedirs(cdc_dir, exist_ok=True)

# Initial load (I) at T-24 months
t0 = datetime.datetime.combine(AS_OF - timedelta(days=730), datetime.time(6, 0))
tables = {
    "fund":           funds,
    "property":       properties,
    "unit":           units,
    "tenant":         tenants,
    "commlease":      leases,
    "lease_charges":  lease_charges[:len(lease_charges)//2],  # first 12 months
    "gl_actuals":     gl_actuals[:len(gl_actuals)//2],
}

file_counts = defaultdict(int)
row_counts = defaultdict(int)
for tbl, recs in tables.items():
    n, fn = write_cdc_file(recs, tbl, "I", t0, cdc_dir)
    file_counts[tbl] += 1
    row_counts[tbl] += n

# Incremental updates (U) and remaining charges/gl over next 12 months
t_mid = t0 + timedelta(days=365)
for tbl, recs in {"lease_charges": lease_charges[len(lease_charges)//2:],
                  "gl_actuals": gl_actuals[len(gl_actuals)//2:]}.items():
    n, fn = write_cdc_file(recs, tbl, "I", t_mid, cdc_dir)
    file_counts[tbl] += 1
    row_counts[tbl] += n

# Simulate ~20 lease updates (rent changes, term extensions), spread over 12 months
updated_leases = random.sample(leases, min(20, len(leases)))
for i, l in enumerate(updated_leases):
    t_upd = t0 + timedelta(days=400 + i * 15)  # spread across ~300 days
    l_copy = dict(l)
    l_copy["monthly_rent"] = round(l_copy["monthly_rent"] * random.uniform(1.02, 1.06), 2)
    l_copy["annual_base_rent"] = round(l_copy["monthly_rent"] * 12, 2)
    n, fn = write_cdc_file([l_copy], "commlease", "U", t_upd, cdc_dir)
    file_counts["commlease"] += 1
    row_counts["commlease"] += n

print("CDC files written to landing/erp_cdc:")
for tbl in sorted(file_counts):
    print(f"  {tbl}: {file_counts[tbl]} files, {row_counts[tbl]:,} rows")
print(f"Total files: {sum(file_counts.values())}")

In [0]:
DEAL_STAGES = ["Screening", "LOI", "Due Diligence", "Under Contract", "Closed", "Dead"]

deals = []
for i in range(1, 21):  # 20 deals; GP-ACQ-014 is guaranteed
    deal_type = "ACQ" if i <= 15 else "DSP"
    deal_id = f"GP-{deal_type}-{i:03d}"
    msa_name = random.choice(list(MSAS.keys()))
    msa = MSAS[msa_name]
    lat, lon = _scatter(msa["lat"], msa["lon"], msa["r"])
    ptype = random.choice(PROPERTY_TYPES)
    ask = round(random.uniform(10_000_000, 120_000_000), -5)
    cap = round(random.uniform(0.045, 0.085), 4)
    deals.append({
        "deal_id": deal_id,
        "deal_type": "Acquisition" if deal_type == "ACQ" else "Disposition",
        "property_type": ptype,
        "market": msa_name,
        "state": msa["state"],
        "address": _street_addr(),
        "lat": lat, "lon": lon,
        "stage": random.choice(DEAL_STAGES[:4]) if deal_id != "GP-ACQ-014" else "Due Diligence",
        "ask_price": ask,
        "noi": round(ask * cap, 2),
        "cap_rate": cap,
        "irr": round(random.uniform(0.08, 0.18), 4),
        "sqft": random.randint(20_000, 300_000),
        "year_built": random.randint(1985, 2024),
    })

# Deal comps (2-4 per deal)
deal_comps = []
for d in deals:
    for c in range(random.randint(2, 4)):
        deal_comps.append({
            "comp_id": f"{d['deal_id']}-COMP-{c+1}",
            "deal_id": d["deal_id"],
            "comp_address": _street_addr(),
            "comp_market": d["market"],
            "comp_price": round(d["ask_price"] * random.uniform(0.7, 1.3), -5),
            "comp_cap_rate": round(d["cap_rate"] + random.uniform(-0.01, 0.01), 4),
            "comp_sqft": d["sqft"] + random.randint(-30_000, 30_000),
            "sale_date": (AS_OF - timedelta(days=random.randint(30, 720))).isoformat(),
        })

# Write deals as CDC
for tbl, recs in {"deal": deals, "deal_comps": deal_comps}.items():
    write_cdc_file(recs, tbl, "I", t0, cdc_dir)

print(f"Deals: {len(deals)}  |  Deal comps: {len(deal_comps)}")
acq14 = next(d for d in deals if d['deal_id'] == 'GP-ACQ-014')
print(f"GP-ACQ-014: {acq14['property_type']} in {acq14['market']}, ask ${acq14['ask_price']:,.0f}, stage={acq14['stage']}")

In [0]:
# Risk-weighted subset: leases expiring within 36 months of AS_OF + top 25 by ABR
# Use 24-month expiry window (the plan says 36 months but sizes the ERP for ~60-100 PDFs;
# with 391 leases, 36 months yields 221 which is too many). 24 months + top-25 ABR lands ~80-100.
cutoff_date = AS_OF + timedelta(days=24 * 30)  # ~24 months

_all_expiring = [l for l in leases if date.fromisoformat(l["lease_end"]) <= cutoff_date]
# Cap expiring set: take the soonest-expiring ~65 leases so total lands in ~60-100
_all_expiring.sort(key=lambda l: l["lease_end"])
expiring_set = {l["lease_id"] for l in _all_expiring[:65]}
top25_set    = {l["lease_id"] for l in sorted(leases, key=lambda x: x["annual_base_rent"], reverse=True)[:25]}
pdf_lease_ids = expiring_set | top25_set
pdf_leases    = [l for l in leases if l["lease_id"] in pdf_lease_ids]

# ~10% will deliberately disagree with ERP (discrepancies for detection)
n_discrepant = max(1, len(pdf_leases) // 10)
discrepant_ids = set(random.sample([l["lease_id"] for l in pdf_leases], n_discrepant))

print(f"PDF lease subset: {len(pdf_leases)} leases")
print(f"  Expiring <=36mo: {len(expiring_set)}  |  Top-25 ABR: {len(top25_set)}  |  Overlap: {len(expiring_set & top25_set)}")
print(f"  Deliberate ERP discrepancies: {n_discrepant}")

In [0]:
lease_dir = f"{VOL_LANDING}/docs/leases"
gt_dir = f"{VOL_ARTIFACTS}/ground_truth/leases"
os.makedirs(lease_dir, exist_ok=True)
os.makedirs(gt_dir, exist_ok=True)

def _make_lease_pdf(lease, tenant, prop, discrepant):
    """Generate a synthetic lease PDF and return ground-truth dict."""
    pdf = FPDF()
    pdf.set_auto_page_break(auto=True, margin=15)
    pdf.add_page()
    pdf.set_font("Helvetica", "B", 16)
    pdf.cell(0, 10, "COMMERCIAL LEASE AGREEMENT", align="C", new_x="LMARGIN", new_y="NEXT")
    pdf.ln(5)
    pdf.set_font("Helvetica", "", 10)
    w = pdf.w - pdf.l_margin - pdf.r_margin  # usable width

    # Ground truth values
    gt = dict(lease)  # copy all fields
    gt["tenant_name"] = tenant["tenant_name"]
    gt["property_address"] = f"{prop['address']}, {prop['city']}, {prop['state']}"
    gt["is_discrepant"] = discrepant

    # If discrepant, shift end date and/or rent in the PDF vs ERP
    pdf_end = lease["lease_end"]
    pdf_rent = lease["monthly_rent"]
    if discrepant:
        shift = random.choice(["end_date", "rent", "both"])
        if shift in ("end_date", "both"):
            d = date.fromisoformat(lease["lease_end"]) + timedelta(days=random.choice([90, 180, -90]))
            pdf_end = d.isoformat()
            gt["discrepancy_type"] = "end_date" if shift == "end_date" else "both"
        if shift in ("rent", "both"):
            pdf_rent = round(lease["monthly_rent"] * random.uniform(0.90, 1.10), 2)
            gt["discrepancy_type"] = gt.get("discrepancy_type", "rent")
        gt["pdf_lease_end"] = pdf_end
        gt["pdf_monthly_rent"] = pdf_rent

    # Lease body
    lines = [
        f"Lease ID: {lease['lease_id']}",
        f"Date: {lease['lease_start']}",
        "",
        f"LANDLORD: Granite Peak Real Estate Partners ({prop['fund_id']})",
        f"TENANT: {tenant['tenant_name']} (ID: {tenant['tenant_id']})",
        "",
        f"PREMISES: {prop['address']}, Unit {next(u['unit_number'] for u in units if u['unit_id']==lease['unit_id'])},"
        f" {prop['city']}, {prop['state']} {prop['zip_code']}",
        f"Property: {prop['property_name']} ({prop['property_type']})",
        "",
        "ARTICLE 1 - TERM",
        f"Commencement Date: {lease['lease_start']}",
        f"Expiration Date: {pdf_end}",
        "",
        "ARTICLE 2 - RENT",
        f"Monthly Base Rent: ${pdf_rent:,.2f}",
        f"Annual Base Rent: ${pdf_rent * 12:,.2f}",
        f"Escalation: {lease['escalation_type']}"
        + (f" at {lease['escalation_rate']*100:.1f}% per annum" if lease['escalation_rate'] else " (Consumer Price Index)"),
        f"Security Deposit: ${lease['security_deposit']:,.2f}",
    ]

    if lease["renewal_option_count"] > 0:
        lines += ["", "ARTICLE 3 - RENEWAL OPTIONS",
                  f"Tenant shall have {lease['renewal_option_count']} option(s) to renew for successive"
                  f" {random.choice([3,5])} year terms upon 180 days written notice."]
    if lease["has_co_tenancy"]:
        lines += ["", "ARTICLE 4 - CO-TENANCY", f"{lease['co_tenancy_terms']}."]
    if lease["has_termination_right"]:
        lines += ["", "ARTICLE 5 - EARLY TERMINATION",
                  f"Tenant may terminate this lease on or after {lease['termination_right_date']}"
                  f" upon {random.choice([90,120,180])} days notice and payment of an early termination fee"
                  f" equal to {random.choice([3,6])} months' rent."]
    if lease["has_exclusive_use"]:
        lines += ["", "ARTICLE 6 - EXCLUSIVE USE", f"{lease['exclusive_use_terms']}."]

    lines += ["", "", "SIGNATURES",
              "_________________________          _________________________",
              "Landlord                           Tenant"]

    for line in lines:
        pdf.multi_cell(w, 5, line)

    fname = f"{lease['lease_id']}.pdf"
    pdf.output(os.path.join(lease_dir, fname))
    return gt

# Generate all lease PDFs
ground_truth_leases = []
for lease in pdf_leases:
    tenant = next(t for t in tenants if t["tenant_id"] == lease["tenant_id"])
    prop   = next(p for p in properties if p["property_id"] == lease["property_id"])
    gt = _make_lease_pdf(lease, tenant, prop, lease["lease_id"] in discrepant_ids)
    ground_truth_leases.append(gt)

# Write ground truth
with open(os.path.join(gt_dir, "lease_ground_truth.json"), "w") as f:
    json.dump(ground_truth_leases, f, indent=2, default=str)

print(f"Lease PDFs written: {len(pdf_leases)} to {lease_dir}")
print(f"Ground truth written: {len(ground_truth_leases)} records to {gt_dir}/lease_ground_truth.json")
print(f"Discrepant PDFs: {sum(1 for g in ground_truth_leases if g.get('is_discrepant'))}")

In [0]:
deal_doc_dir = f"{VOL_LANDING}/docs/deals"
gt_deal_dir  = f"{VOL_ARTIFACTS}/ground_truth/deals"
os.makedirs(deal_doc_dir, exist_ok=True)
os.makedirs(gt_deal_dir, exist_ok=True)

def _make_deal_om(deal, deal_leases_for_om):
    """Generate an offering memorandum PDF for a deal."""
    pdf = FPDF()
    pdf.set_auto_page_break(auto=True, margin=15)
    pdf.add_page()
    pdf.set_font("Helvetica", "B", 18)
    pdf.cell(0, 12, "CONFIDENTIAL OFFERING MEMORANDUM", align="C", new_x="LMARGIN", new_y="NEXT")
    pdf.ln(8)
    pdf.set_font("Helvetica", "B", 14)
    pdf.cell(0, 10, f"{deal['address']}, {deal['market']}, {deal['state']}", align="C", new_x="LMARGIN", new_y="NEXT")
    pdf.ln(5)
    pdf.set_font("Helvetica", "", 10)
    w = pdf.w - pdf.l_margin - pdf.r_margin

    lines = [
        f"Deal ID: {deal['deal_id']}",
        f"Property Type: {deal['property_type']}",
        f"Size: {deal['sqft']:,} SF  |  Year Built: {deal['year_built']}",
        f"Asking Price: ${deal['ask_price']:,.0f}",
        f"Net Operating Income: ${deal['noi']:,.0f}",
        f"Cap Rate: {deal['cap_rate']*100:.2f}%",
        f"Projected IRR: {deal['irr']*100:.1f}%",
        "",
        "INVESTMENT HIGHLIGHTS",
        f"  - {deal['property_type']} asset in {deal['market']} MSA",
        f"  - Strong in-place cash flow with NOI of ${deal['noi']:,.0f}",
        f"  - Below-replacement-cost basis at ${deal['ask_price']/max(deal['sqft'],1):,.0f}/SF",
    ]

    # In-place lease summary (synthetic tenants for the deal)
    if deal_leases_for_om:
        lines += ["", "IN-PLACE LEASE SUMMARY", ""]
        total_abr = 0
        for dl in deal_leases_for_om:
            lines.append(f"  Tenant: {dl['tenant_name']}  |  ABR: ${dl['abr']:,.0f}"
                         f"  |  Expiry: {dl['lease_end']}")
            if dl.get("renewal_options", 0) > 0:
                lines.append(f"    Renewal: {dl['renewal_options']} option(s)")
            if dl.get("has_co_tenancy"):
                lines.append(f"    Co-tenancy: {dl['co_tenancy_terms']}")
            total_abr += dl["abr"]
        lines += [f"  Total In-Place ABR: ${total_abr:,.0f}"]

    lines += ["", "", "This memorandum is confidential and for the sole use of the intended recipient."]

    for line in lines:
        pdf.multi_cell(w, 5, line)

    pdf.output(os.path.join(deal_doc_dir, f"{deal['deal_id']}_OM.pdf"))
    return {"deal_id": deal["deal_id"], "doc_type": "offering_memorandum",
            "ask_price": deal["ask_price"], "noi": deal["noi"], "cap_rate": deal["cap_rate"],
            "in_place_leases": deal_leases_for_om}

def _make_deal_rent_roll(deal, deal_leases_for_rr):
    """Generate a rent roll PDF for a deal."""
    pdf = FPDF()
    pdf.set_auto_page_break(auto=True, margin=15)
    pdf.add_page("L")  # landscape
    pdf.set_font("Helvetica", "B", 14)
    pdf.cell(0, 10, f"RENT ROLL - {deal['deal_id']}", align="C", new_x="LMARGIN", new_y="NEXT")
    pdf.ln(3)
    pdf.set_font("Helvetica", "B", 8)
    cols = ["Tenant", "Suite", "SF", "Start", "End", "Mo Rent", "ABR"]
    cw = [50, 18, 22, 28, 28, 30, 30]  # total 206 fits landscape
    for c, cwidth in zip(cols, cw):
        pdf.cell(cwidth, 7, c, border=1)
    pdf.ln()
    pdf.set_font("Helvetica", "", 8)
    for dl in deal_leases_for_rr:
        vals = [dl["tenant_name"][:18], dl.get("suite", "A"), f"{dl['sqft']:,}",
                dl["lease_start"], dl["lease_end"],
                f"${dl['monthly_rent']:,.0f}", f"${dl['abr']:,.0f}"]
        for v, cwidth in zip(vals, cw):
            pdf.cell(cwidth, 6, str(v), border=1)
        pdf.ln()
    pdf.output(os.path.join(deal_doc_dir, f"{deal['deal_id']}_RentRoll.pdf"))
    return {"deal_id": deal["deal_id"], "doc_type": "rent_roll", "tenant_count": len(deal_leases_for_rr)}

# For each deal, create synthetic in-place leases (3-8 tenants)
deal_gt = []
for deal in deals[:13]:  # ~13 deals get documents (OM + rent roll)
    n_tenants = random.randint(3, 8)
    deal_tenants = []
    for t in range(n_tenants):
        tname = _tenant_name(deal["property_type"])
        start = (AS_OF - timedelta(days=random.randint(365, 2500))).isoformat()
        end   = (AS_OF + timedelta(days=random.randint(180, 2500))).isoformat()
        abr   = round(random.uniform(50_000, 800_000), 2)
        renewal = random.choice([0, 1, 2])
        has_co = random.random() < 0.2
        deal_tenants.append({
            "tenant_name": tname,
            "suite": f"{random.randint(1,5)}{chr(65+t)}",
            "sqft": random.randint(2000, 30000),
            "lease_start": start, "lease_end": end,
            "monthly_rent": round(abr / 12, 2), "abr": abr,
            "renewal_options": renewal,
            "has_co_tenancy": has_co,
            "co_tenancy_terms": f"Anchor at {random.randint(60,80)}% occupancy" if has_co else None,
        })
    gt_om = _make_deal_om(deal, deal_tenants)
    gt_rr = _make_deal_rent_roll(deal, deal_tenants)
    deal_gt.extend([gt_om, gt_rr])

with open(os.path.join(gt_deal_dir, "deal_ground_truth.json"), "w") as f:
    json.dump(deal_gt, f, indent=2, default=str)

print(f"Deal documents written: {len(deal_gt)} files ({len(deal_gt)//2} OMs + {len(deal_gt)//2} rent rolls)")
print(f"Deal ground truth: {gt_deal_dir}/deal_ground_truth.json")

In [0]:
print("="*60)
print("SYNTHETIC DATA GENERATION SUMMARY")
print("="*60)
print(f"Properties:       {len(properties):>6}")
print(f"Units:            {len(units):>6}")
print(f"Tenants:          {len(tenants):>6}")
print(f"Leases:           {len(leases):>6}")
print(f"Lease charges:    {len(lease_charges):>6,}")
print(f"GL actuals:       {len(gl_actuals):>6,}")
print(f"Deals:            {len(deals):>6}")
print(f"Deal comps:       {len(deal_comps):>6}")
print(f"Lease PDFs:       {len(pdf_leases):>6}")
print(f"Deal docs:        {len(deal_gt):>6}")
print(f"CDC files:        {sum(file_counts.values()) + 2:>6}")  # +2 for deal/deal_comps
print("="*60)
print(f"\nSample property:")
print(json.dumps(properties[0], indent=2, default=str))
print(f"\nSample lease:")
print(json.dumps(leases[0], indent=2, default=str))
print(f"\nDeal GP-ACQ-014:")
print(json.dumps(acq14, indent=2, default=str))
print(f"\nLanding volume contents:")
for item in dbutils.fs.ls(f"/Volumes/{catalog}/{schema}/landing/erp_cdc"):
    print(f"  {item.name}  ({item.size:,} bytes)")